# VIA2 SOTA Vision Model Verification

Google Colab T4 GPU에서 5개 SOTA 비전 모델의 로드/추론/메모리를 검증합니다.

모델 목록:
1. Florence-2 (microsoft/Florence-2-base)
2. Grounding DINO (IDEA-Research/grounding-dino-base)
3. SAM 2 (facebook/sam2-hiera-small)
4. DINOv2 (facebook/dinov2-base)
5. Depth-Anything-V2 (depth-anything/Depth-Anything-V2-Small-hf)

In [ ]:
# Install required packages
!pip install -q transformers accelerate timm einops
!pip install -q torch torchvision
!pip install -q Pillow requests supervision
print("Installation complete.")

In [ ]:
import torch
import gc
import requests
from PIL import Image
from io import BytesIO

# GPU memory helper
def get_gpu_mem():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**3
    return 0.0

# Download sample image once
IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg"
try:
    response = requests.get(IMAGE_URL, timeout=10)
    sample_image = Image.open(BytesIO(response.content)).convert("RGB")
    print(f"Sample image loaded: {sample_image.size}")
except Exception as e:
    print(f"Image download failed: {e}. Using fallback solid color image.")
    sample_image = Image.new("RGB", (320, 320), color=(128, 128, 128))
    print(f"Fallback image created: {sample_image.size}")

# Results collection
results = {}
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 1. Florence-2

- **HuggingFace**: `florence-community/Florence-2-base` (transformers 5.x native) / `microsoft/Florence-2-base` (transformers 4.x fallback)
- **VIA2 역할**: Material Agent — 소재 분류 및 이미지 캡셔닝 (Step 16)

In [ ]:
result_florence2 = {"loaded": False, "inferred": False, "gpu_mem_gb": 0.0}
try:
    torch.cuda.empty_cache()
    mem_before = get_gpu_mem()
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    from transformers import AutoProcessor
    try:
        # transformers 5.x: native Florence-2 support
        from transformers import Florence2ForConditionalGeneration
        model = Florence2ForConditionalGeneration.from_pretrained(
            "florence-community/Florence-2-base",
            dtype=torch.float16,
            device_map="auto"
        )
        processor = AutoProcessor.from_pretrained("florence-community/Florence-2-base")
        print("Florence-2: using native checkpoint (transformers 5.x)")
    except (ImportError, Exception) as load_e:
        print(f"Native checkpoint failed: {load_e}\nFalling back to microsoft/Florence-2-base...")
        from transformers import AutoModelForCausalLM
        model = AutoModelForCausalLM.from_pretrained(
            "microsoft/Florence-2-base",
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
            attn_implementation="eager"
        )
        processor = AutoProcessor.from_pretrained("microsoft/Florence-2-base", trust_remote_code=True)
        print("Florence-2: using microsoft checkpoint (trust_remote_code)")
    result_florence2["loaded"] = True
    mem_after = get_gpu_mem()
    print(f"Florence-2 loaded. GPU mem: {mem_after:.2f} GB")
    
    inputs = processor(text="<CAPTION>", images=sample_image, return_tensors="pt").to(model.device, torch.float16)
    output = model.generate(**inputs, max_new_tokens=50)
    caption = processor.decode(output[0], skip_special_tokens=True)
    print(f"Caption: {caption}")
    result_florence2["inferred"] = True
    result_florence2["gpu_mem_gb"] = mem_after - mem_before
except Exception as e:
    print(f"Florence-2 ERROR: {e}")
    import traceback; traceback.print_exc()
finally:
    try:
        del model, processor
    except: pass
    torch.cuda.empty_cache(); gc.collect()

results["Florence-2"] = result_florence2
print(f"Florence-2 result: {result_florence2}")

## 2. Grounding DINO

- **HuggingFace**: `IDEA-Research/grounding-dino-base`
- **VIA2 역할**: ROI Agent — 텍스트 프롬프트로 결함/부품 영역 검출 (Step 17)

In [ ]:
result_grounding_dino = {"loaded": False, "inferred": False, "gpu_mem_gb": 0.0}
try:
    torch.cuda.empty_cache()
    mem_before = get_gpu_mem()
    
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
    processor = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-base")
    model = AutoModelForZeroShotObjectDetection.from_pretrained("IDEA-Research/grounding-dino-base")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    result_grounding_dino["loaded"] = True
    mem_after = get_gpu_mem()
    print(f"Grounding DINO loaded. GPU mem: {mem_after:.2f} GB")
    
    text_prompt = "a dog. a cat. a person."
    inputs = processor(images=sample_image, text=text_prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    target_sizes = torch.tensor([sample_image.size[::-1]])
    results_det = processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids, threshold=0.3, text_threshold=0.3, target_sizes=target_sizes
    )
    print(f"Detections: {len(results_det[0]['boxes'])} boxes")
    result_grounding_dino["inferred"] = True
    result_grounding_dino["gpu_mem_gb"] = mem_after - mem_before
except Exception as e:
    print(f"Grounding DINO ERROR: {e}")
    import traceback; traceback.print_exc()
finally:
    try:
        del model, processor
    except: pass
    torch.cuda.empty_cache(); gc.collect()

results["Grounding DINO"] = result_grounding_dino
print(f"Grounding DINO result: {result_grounding_dino}")

## 3. SAM 2

- **HuggingFace**: `facebook/sam2-hiera-small`
- **VIA2 역할**: ROI Agent — 결함 영역 정밀 세그멘테이션 (Step 17)

In [ ]:
result_sam2 = {"loaded": False, "inferred": False, "gpu_mem_gb": 0.0}
try:
    torch.cuda.empty_cache()
    mem_before = get_gpu_mem()
    
    from transformers import pipeline
    pipe = pipeline("mask-generation", model="facebook/sam2-hiera-small", device=0 if torch.cuda.is_available() else -1)
    result_sam2["loaded"] = True
    mem_after = get_gpu_mem()
    print(f"SAM 2 loaded. GPU mem: {mem_after:.2f} GB")
    outputs = pipe(sample_image, points_per_batch=64)
    print(f"SAM 2 masks generated: {len(outputs.get('masks', []))}")
    result_sam2["inferred"] = True
    result_sam2["gpu_mem_gb"] = mem_after - mem_before
except Exception as e:
    print(f"SAM 2 ERROR: {e}")
    import traceback; traceback.print_exc()
finally:
    try:
        del pipe
    except: pass
    torch.cuda.empty_cache(); gc.collect()

results["SAM 2"] = result_sam2
print(f"SAM 2 result: {result_sam2}")

## 4. DINOv2

- **HuggingFace**: `facebook/dinov2-base`
- **VIA2 역할**: Material Agent + Decision Agent — 특징 벡터 기반 소재 유사도 비교 (Step 16, 32)

In [ ]:
result_dinov2 = {"loaded": False, "inferred": False, "gpu_mem_gb": 0.0}
try:
    torch.cuda.empty_cache()
    mem_before = get_gpu_mem()
    
    from transformers import AutoImageProcessor, AutoModel
    processor = AutoImageProcessor.from_pretrained("facebook/dinov2-base")
    model = AutoModel.from_pretrained("facebook/dinov2-base")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    result_dinov2["loaded"] = True
    mem_after = get_gpu_mem()
    print(f"DINOv2 loaded. GPU mem: {mem_after:.2f} GB")
    
    inputs = processor(images=sample_image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    embedding = outputs.last_hidden_state[:, 0, :]
    print(f"DINOv2 embedding shape: {embedding.shape}")
    result_dinov2["inferred"] = True
    result_dinov2["gpu_mem_gb"] = mem_after - mem_before
except Exception as e:
    print(f"DINOv2 ERROR: {e}")
    import traceback; traceback.print_exc()
finally:
    try:
        del model, processor
    except: pass
    torch.cuda.empty_cache(); gc.collect()

results["DINOv2"] = result_dinov2
print(f"DINOv2 result: {result_dinov2}")

## 5. Depth-Anything-V2

- **HuggingFace**: `depth-anything/Depth-Anything-V2-Small-hf`
- **VIA2 역할**: Depth Agent — 소재 표면 깊이 맵 추정 (Step 15)

In [ ]:
result_depth = {"loaded": False, "inferred": False, "gpu_mem_gb": 0.0}
try:
    torch.cuda.empty_cache()
    mem_before = get_gpu_mem()
    
    from transformers import AutoImageProcessor, AutoModelForDepthEstimation
    processor = AutoImageProcessor.from_pretrained("depth-anything/Depth-Anything-V2-Small-hf")
    model = AutoModelForDepthEstimation.from_pretrained("depth-anything/Depth-Anything-V2-Small-hf")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    result_depth["loaded"] = True
    mem_after = get_gpu_mem()
    print(f"Depth-Anything-V2 loaded. GPU mem: {mem_after:.2f} GB")
    
    inputs = processor(images=sample_image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    depth_map = outputs.predicted_depth
    print(f"Depth map shape: {depth_map.shape}")
    result_depth["inferred"] = True
    result_depth["gpu_mem_gb"] = mem_after - mem_before
except Exception as e:
    print(f"Depth-Anything-V2 ERROR: {e}")
    import traceback; traceback.print_exc()
finally:
    try:
        del model, processor
    except: pass
    torch.cuda.empty_cache(); gc.collect()

results["Depth-Anything-V2"] = result_depth
print(f"Depth-Anything-V2 result: {result_depth}")

## Summary

In [ ]:
print("\n" + "="*60)
print("VERIFICATION SUMMARY")
print("="*60)
print(f"{'Model':<30} {'Loaded':<10} {'Inferred':<12} {'GPU Mem (GB)':<15}")
print("-"*67)
for name, res in results.items():
    print(f"{name:<30} {str(res['loaded']):<10} {str(res['inferred']):<12} {res['gpu_mem_gb']:<15.2f}")
print("="*60)
print("\nNote: GPU Mem shows delta after model load (before inference).")
print("Run on Colab T4 to get actual measurements.")